In [ ]:
Dataset Download: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

In [1]:
import pandas as pd

# Load Kaggle dataset
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Rename columns
df = df.rename(columns={
    "customerID": "CustomerID",
    "gender": "Gender",
    "tenure": "Tenure",
    "MonthlyCharges": "MonthlyCharges",
    "TotalCharges": "TotalCharges",
    "Contract": "ContractType",
    "PaymentMethod": "PaymentMethod",
    "InternetService": "InternetService",
    "Churn": "Churn"
})

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Create Age column (dataset doesn't have one)
import numpy as np
np.random.seed(42)
df["Age"] = np.random.randint(18, 70, len(df))

# Create SupportTickets column (dataset doesn't have one)
df["SupportTickets"] = np.random.randint(0, 8, len(df))

# Keep only required columns
df = df[
    [
        "CustomerID",
        "Gender",
        "Age",
        "Tenure",
        "MonthlyCharges",
        "TotalCharges",
        "ContractType",
        "PaymentMethod",
        "InternetService",
        "SupportTickets",
        "Churn",
    ]
]

df.to_csv("customer_churn.csv", index=False)

print(df.head())

   CustomerID  Gender  Age  Tenure  MonthlyCharges  TotalCharges  \
0  7590-VHVEG  Female   56       1           29.85         29.85   
1  5575-GNVDE    Male   69      34           56.95       1889.50   
2  3668-QPYBK    Male   46       2           53.85        108.15   
3  7795-CFOCW    Male   32      45           42.30       1840.75   
4  9237-HQITU  Female   60       2           70.70        151.65   

     ContractType              PaymentMethod InternetService  SupportTickets  \
0  Month-to-month           Electronic check             DSL               4   
1        One year               Mailed check             DSL               4   
2  Month-to-month               Mailed check             DSL               2   
3        One year  Bank transfer (automatic)             DSL               2   
4  Month-to-month           Electronic check     Fiber optic               7   

  Churn  
0    No  
1    No  
2   Yes  
3    No  
4   Yes  


In [2]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("customer_churn.csv")

print(df.head())

print(df.info())

print(df.isnull().sum())

print("Duplicate Rows:", df.duplicated().sum())

print(df["Churn"].value_counts())

# -----------------------------
# Remove CustomerID
# -----------------------------
df = df.drop("CustomerID", axis=1)

# -----------------------------
# Features and Target
# -----------------------------
X = df.drop("Churn", axis=1)
y = df["Churn"]

# -----------------------------
# Train Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -----------------------------
# Numerical Columns
# -----------------------------
numeric_features = [
    "Age",
    "Tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SupportTickets"
]

# -----------------------------
# Categorical Columns
# -----------------------------
categorical_features = [
    "Gender",
    "ContractType",
    "PaymentMethod",
    "InternetService"
]

# -----------------------------
# Numerical Pipeline
# -----------------------------
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

# -----------------------------
# Categorical Pipeline
# -----------------------------
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

# -----------------------------
# Combine Pipelines
# -----------------------------
preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_features
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_features
    )
])

# -----------------------------
# Final Pipeline
# -----------------------------
model_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )
    )
])

# -----------------------------
# Train Model
# -----------------------------
model_pipeline.fit(X_train, y_train)

# -----------------------------
# Prediction
# -----------------------------
y_pred = model_pipeline.predict(X_test)

# -----------------------------
# Evaluation
# -----------------------------
print("\nAccuracy")

print(accuracy_score(y_test, y_pred))

print("\nConfusion Matrix")

print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")

print(classification_report(y_test, y_pred))

# -----------------------------
# Save Pipeline
# -----------------------------
joblib.dump(
    model_pipeline,
    "customer_churn_pipeline.pkl"
)

print("\nPipeline Saved Successfully!")

# -----------------------------
# Load Pipeline
# -----------------------------
loaded_pipeline = joblib.load(
    "customer_churn_pipeline.pkl"
)

# -----------------------------
# New Customer Prediction
# -----------------------------
new_customer = pd.DataFrame({

    "Gender": ["Female"],

    "Age": [32],

    "Tenure": [5],

    "MonthlyCharges": [85.0],

    "TotalCharges": [425.0],

    "ContractType": ["Month-to-month"],

    "PaymentMethod": ["Electronic check"],

    "InternetService": ["Fiber optic"],

    "SupportTickets": [3]

})

prediction = loaded_pipeline.predict(new_customer)

print("\nPrediction for New Customer:")

print(prediction[0])

   CustomerID  Gender  Age  Tenure  MonthlyCharges  TotalCharges  \
0  7590-VHVEG  Female   56       1           29.85         29.85   
1  5575-GNVDE    Male   69      34           56.95       1889.50   
2  3668-QPYBK    Male   46       2           53.85        108.15   
3  7795-CFOCW    Male   32      45           42.30       1840.75   
4  9237-HQITU  Female   60       2           70.70        151.65   

     ContractType              PaymentMethod InternetService  SupportTickets  \
0  Month-to-month           Electronic check             DSL               4   
1        One year               Mailed check             DSL               4   
2  Month-to-month               Mailed check             DSL               2   
3        One year  Bank transfer (automatic)             DSL               2   
4  Month-to-month           Electronic check     Fiber optic               7   

  Churn  
0    No  
1    No  
2   Yes  
3    No  
4   Yes  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 